In [1]:
import numpy as np
import bgshr
import pandas
import os

In [2]:
# Load centromere regions

def get_centromere(chrom):
    """
    Get upper and lower bounds of the `acen` annotation from 
    cytoBand.txt.
    """ 
    band_df = pandas.read_csv("../data/cytoBand.txt", sep="\t",
        names=["chrom", "start", "end", "band", "stain"])
    sub_df = band_df[band_df["chrom"] == f"chr{chrom}"]
    start = np.min(sub_df[sub_df["stain"] == "acen"]["start"])
    end = np.max(sub_df[sub_df["stain"] == "acen"]["end"])
    return (start, end)

In [5]:
# Build tables with expected, observed diversity and other maps
# For the equilibrium/granular Ne demography

scale = 1e4
chroms = list(range(1, 23))
mut_models = [
    "roulette",
    "gnomad",
    "carlson"]
cons_models = [
    "merged_cds_regulatory",
    "split_cds_regulatory",
    "merged_cds_phastcons",
    "split_cds_phastcons"]
demography = "equilibrium_granular_Ne"

for chrom in chroms:
    for mut_model in mut_models:
        for cons_model in cons_models:
            in_fname = (f"../local_models/{demgoraphy}/{cons_model}/{mut_model}/{cons_model}_{mut_model}_chr{chrom}.csv.gz")
            df = pandas.read_csv(in_fname)
            # load diversity data
            df_pi = pandas.read_csv(f"../data/pi_tables/{mut_model}/pi_tbl_YRI_chr{chrom}.csv.gz")
            assert len(df_pi["avg_pi"]) == len(df["exp_pi"])
            assert np.all(np.array(df_pi["num_sites"]) == np.array(df["num_sites"]))
            df["avg_pi"] = df_pi["avg_pi"]
            # drop windows within the centromere
            cen_start, cen_end = get_centromere(chrom)
            df = df[np.logical_not((df["chromStart"] >= cen_start) 
                                   & (df["chromEnd"] <= cen_end))]
            # drop the HLA region from chromosome 6
            if chrom == 6:
                # determined with UCSC genome browser
                hla_start, hla_end = 28500000, 33500000
                df = df[np.logical_not((df["chromStart"] >= hla_start) 
                                       & (df["chromEnd"] <= hla_end))]
            # re-order fields
            df_scaled = bgshr.Util.scale_genome_table(df, scale)
            df_scaled = df_scaled.loc[:, [
                "chrom", 
                "chromStart",
                "chromEnd",
                "num_sites",
                "del_sites",
                "avg_mut",
                "del_mut",
                "avg_rec",
                "B",
                "exp_del_pi",
                "exp_pi",
                "avg_pi"
            ]]
            # drop empty windows
            df_scaled = df_scaled[df_scaled["num_sites"] > 0]
            out_fname = (f"../models/{demography}/{cons_model}/{mut_model}/B_tbl_YRI_chr{chrom}_10kb.csv.gz")
            df_scaled.to_csv(out_fname, index=False, na_rep=0.0, compression="gzip")
            print(chrom, mut_model, cons_model)

1 roulette split_cds_regulatory
1 gnomad split_cds_regulatory
1 carlson split_cds_regulatory
2 roulette split_cds_regulatory
2 gnomad split_cds_regulatory
2 carlson split_cds_regulatory
3 roulette split_cds_regulatory
3 gnomad split_cds_regulatory
3 carlson split_cds_regulatory
4 roulette split_cds_regulatory
4 gnomad split_cds_regulatory
4 carlson split_cds_regulatory
5 roulette split_cds_regulatory
5 gnomad split_cds_regulatory
5 carlson split_cds_regulatory
6 roulette split_cds_regulatory
6 gnomad split_cds_regulatory
6 carlson split_cds_regulatory
7 roulette split_cds_regulatory
7 gnomad split_cds_regulatory
7 carlson split_cds_regulatory
8 roulette split_cds_regulatory
8 gnomad split_cds_regulatory
8 carlson split_cds_regulatory
9 roulette split_cds_regulatory
9 gnomad split_cds_regulatory
9 carlson split_cds_regulatory
10 roulette split_cds_regulatory
10 gnomad split_cds_regulatory
10 carlson split_cds_regulatory
11 roulette split_cds_regulatory
11 gnomad split_cds_regulatory
11 

In [14]:
# Build tables with expected, observed diversity and other maps
# For the equilibrium/single Ne demography

scale = 1e4
chroms = list(range(1, 23))
mut_models = [
    "roulette",
    "gnomad",
    "carlson"]
cons_models = [
    #"merged_cds_regulatory",
    #"split_cds_regulatory", ]
    "merged_cds_phastcons",
    "split_cds_phastcons"]
demography = "equilibrium_single_Ne"

for chrom in chroms:
    for mut_model in mut_models:
        for cons_model in cons_models:
            in_path = f"../local_models/tables_1kb/{demography}/{cons_model}/{mut_model}"
            out_path = f"../models/tables_10kb/{demography}/{cons_model}/{mut_model}"
            in_fname = f"{in_path}/{cons_model}_{mut_model}_chr{chrom}.csv.gz"
            out_fname = f"{out_path}/B_tbl_YRI_chr{chrom}_10kb.csv.gz"
            if not os.path.exists(out_path):
                !mkdir -p out_path
            df = pandas.read_csv(in_fname)
            # load diversity data
            df_pi = pandas.read_csv(f"../data/pi_tables/{mut_model}/pi_tbl_YRI_chr{chrom}.csv.gz")
            assert len(df_pi["avg_pi"]) == len(df["exp_pi"])
            assert np.all(np.array(df_pi["num_sites"]) == np.array(df["num_sites"]))
            df["avg_pi"] = df_pi["avg_pi"]
            # order columns and drop any columns not specified             
            df = df.loc[:, [
                "chrom", 
                "chromStart",
                "chromEnd",
                "num_sites",
                "del_sites",
                "avg_mut",
                "del_mut",
                "avg_rec",
                "B",
                "exp_del_pi",
                "exp_pi",
                "avg_pi"
            ]]
            # drop windows within the centromere
            cen_start, cen_end = get_centromere(chrom)
            df = df[np.logical_not((df["chromStart"] >= cen_start) 
                                   & (df["chromEnd"] <= cen_end))]
            # drop the HLA region from chromosome 6
            if chrom == 6:
                # determined with UCSC genome browser
                hla_start, hla_end = 28500000, 33500000
                df = df[np.logical_not((df["chromStart"] >= hla_start) 
                                       & (df["chromEnd"] <= hla_end))]
            # scale the table to 10kb window size
            df_scaled = bgshr.Util.scale_genome_table(df, scale)
            # drop empty windows
            df_scaled = df_scaled[df_scaled["num_sites"] > 0]
            df_scaled.to_csv(out_fname, index=False, na_rep=0.0, compression="gzip")
            print(chrom, mut_model, cons_model)

1 roulette merged_cds_phastcons
1 roulette split_cds_phastcons
1 gnomad merged_cds_phastcons
1 gnomad split_cds_phastcons
1 carlson merged_cds_phastcons
1 carlson split_cds_phastcons
2 roulette merged_cds_phastcons
2 roulette split_cds_phastcons
2 gnomad merged_cds_phastcons
2 gnomad split_cds_phastcons
2 carlson merged_cds_phastcons
2 carlson split_cds_phastcons
3 roulette merged_cds_phastcons
3 roulette split_cds_phastcons
3 gnomad merged_cds_phastcons
3 gnomad split_cds_phastcons
3 carlson merged_cds_phastcons
3 carlson split_cds_phastcons
4 roulette merged_cds_phastcons
4 roulette split_cds_phastcons
4 gnomad merged_cds_phastcons
4 gnomad split_cds_phastcons
4 carlson merged_cds_phastcons
4 carlson split_cds_phastcons
5 roulette merged_cds_phastcons
5 roulette split_cds_phastcons
5 gnomad merged_cds_phastcons
5 gnomad split_cds_phastcons
5 carlson merged_cds_phastcons
5 carlson split_cds_phastcons
6 roulette merged_cds_phastcons
6 roulette split_cds_phastcons
6 gnomad merged_cds_ph

In [15]:
# Pure CDS models (10kb scale: fits)

scale = 1e4
chroms = list(range(1, 23))
mut_models = [
    "roulette",
    "gnomad",
    "carlson"]
cons_models = [
    "merged_cds",
    "split_cds"]
demography = "equilibrium_granular_Ne"

for chrom in chroms:
    for mut_model in mut_models:
        for cons_model in cons_models:
            in_path = f"../local_models/tables_10kb/{demography}/{cons_model}/{mut_model}"
            out_path = f"../models/tables_10kb/{demography}/{cons_model}/{mut_model}"
            in_fname = f"{in_path}/{cons_model}_{mut_model}_chr{chrom}_map.csv.gz"
            out_fname = f"{out_path}/B_tbl_YRI_chr{chrom}_10kb.csv.gz"
            if not os.path.exists(out_path):
                print("creating path", out_path)
                os.makedirs(out_path)
            df = pandas.read_csv(in_fname)
            # load diversity data
            df_pi = pandas.read_csv(f"../data/pi_tables/{mut_model}/pi_tbl_YRI_chr{chrom}.csv.gz")
            # scale the pi table to 10kb window size
            df_pi_scaled = bgshr.Util.scale_genome_table(df_pi, scale)
            if len(df_pi_scaled) > len(df):
                df_pi_scaled = df_pi_scaled.loc[:len(df) - 1]
            assert len(df_pi_scaled["avg_pi"]) == len(df["exp_pi"])
            assert np.all(np.array(df_pi_scaled["num_sites"]) == np.array(df["num_sites"]))
            df["avg_pi"] = df_pi_scaled["avg_pi"]
            # order columns and drop any columns not specified             
            df = df.loc[:, [
                "chrom", 
                "chromStart",
                "chromEnd",
                "num_sites",
                "del_sites",
                "avg_mut",
                "del_mut",
                "avg_rec",
                "B",
                "exp_del_pi",
                "exp_pi",
                "avg_pi"
            ]]
            # drop windows within the centromere
            cen_start, cen_end = get_centromere(chrom)
            df = df[np.logical_not((df["chromStart"] >= cen_start) 
                                   & (df["chromEnd"] <= cen_end))]
            # drop the HLA region from chromosome 6
            if chrom == 6:
                # determined with UCSC genome browser
                hla_start, hla_end = 28500000, 33500000
                df = df[np.logical_not((df["chromStart"] >= hla_start) 
                                       & (df["chromEnd"] <= hla_end))]
            # drop empty windows
            df = df[df["num_sites"] > 0]
            df.to_csv(out_fname, index=False, na_rep=0.0, compression="gzip")
            print(chrom, mut_model, cons_model)

creating path ../models/tables_10kb/equilibrium_granular_Ne/merged_cds/roulette
1 roulette merged_cds
creating path ../models/tables_10kb/equilibrium_granular_Ne/split_cds/roulette
1 roulette split_cds
creating path ../models/tables_10kb/equilibrium_granular_Ne/merged_cds/gnomad
1 gnomad merged_cds
creating path ../models/tables_10kb/equilibrium_granular_Ne/split_cds/gnomad
1 gnomad split_cds
creating path ../models/tables_10kb/equilibrium_granular_Ne/merged_cds/carlson
1 carlson merged_cds
creating path ../models/tables_10kb/equilibrium_granular_Ne/split_cds/carlson
1 carlson split_cds
2 roulette merged_cds
2 roulette split_cds
2 gnomad merged_cds
2 gnomad split_cds
2 carlson merged_cds
2 carlson split_cds
3 roulette merged_cds
3 roulette split_cds
3 gnomad merged_cds
3 gnomad split_cds
3 carlson merged_cds
3 carlson split_cds
4 roulette merged_cds
4 roulette split_cds
4 gnomad merged_cds
4 gnomad split_cds
4 carlson merged_cds
4 carlson split_cds
5 roulette merged_cds
5 roulette spli

In [5]:
# Build pure B-tables at the 1kb scale for select models
chroms = list(range(1, 23))
mut_models = ["roulette"]
cons_models = [
    "split_cds_regulatory",
    "split_cds_phastcons"]
demography = "equilibrium_granular_Ne"

for chrom in chroms:
    for mut_model in mut_models:
        for cons_model in cons_models:
            in_path = f"../local_models/tables_1kb/{demography}/{cons_model}/{mut_model}"
            out_path = f"../models/maps_1kb/{demography}/{cons_model}/{mut_model}"
            in_fname = f"{in_path}/{cons_model}_{mut_model}_chr{chrom}.csv.gz"
            out_fname = f"{out_path}/B_map_YRI_chr{chrom}_1kb.csv.gz"
            if not os.path.exists(out_path):
                !mkdir -p out_path
            df = pandas.read_csv(in_fname)
            # Recover focal positions where B was predicted
            starts = np.array(df["chromStart"])
            ends = np.array(df["chromEnd"])
            pos = ((starts + ends) / 2).astype(np.int64)
            df_out = pandas.DataFrame({
                "chrom": df["chrom"],
                "pos": pos,
                "B": df["B"]})
            df_out.to_csv(out_fname, index=False, compression="gzip")
            print(chrom, mut_model, cons_model)

1 roulette split_cds_regulatory
1 roulette split_cds_phastcons
2 roulette split_cds_regulatory
2 roulette split_cds_phastcons
3 roulette split_cds_regulatory
3 roulette split_cds_phastcons
4 roulette split_cds_regulatory
4 roulette split_cds_phastcons
5 roulette split_cds_regulatory
5 roulette split_cds_phastcons
6 roulette split_cds_regulatory
6 roulette split_cds_phastcons
7 roulette split_cds_regulatory
7 roulette split_cds_phastcons
8 roulette split_cds_regulatory
8 roulette split_cds_phastcons
9 roulette split_cds_regulatory
9 roulette split_cds_phastcons
10 roulette split_cds_regulatory
10 roulette split_cds_phastcons
11 roulette split_cds_regulatory
11 roulette split_cds_phastcons
12 roulette split_cds_regulatory
12 roulette split_cds_phastcons
13 roulette split_cds_regulatory
13 roulette split_cds_phastcons
14 roulette split_cds_regulatory
14 roulette split_cds_phastcons
15 roulette split_cds_regulatory
15 roulette split_cds_phastcons
16 roulette split_cds_regulatory
16 roulett